In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle

In [2]:
data=pd.read_csv('Churn_Modelling.csv')
data

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [3]:
## Encode categorical variables
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])


In [4]:
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder(handle_unknown='ignore')
geo_encoded=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [5]:
## split the data into features and the target
x=data.drop('EstimatedSalary',axis=1)
y=data['EstimatedSalary']

In [6]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3)

In [7]:
## load the encoder and scaler 
with  open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)
    
with  open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)
    
with  open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)

### ANN Regression Problem Statement


In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [9]:
x_train

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
9271,9272,15774285,Kentish,649,Spain,0,47,8,110783.28,1,1,1,0
6602,6603,15580872,Chinweike,761,Germany,0,38,1,120530.13,2,1,0,0
1528,1529,15597131,Fu,415,France,1,32,5,145807.59,1,1,1,0
4026,4027,15606641,Beggs,762,Germany,1,56,10,100260.88,3,1,1,1
4922,4923,15751203,Cattaneo,702,France,1,26,5,56738.47,2,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9584,9585,15608588,Mackinlay,563,Germany,1,41,2,100520.92,1,1,1,1
1968,1969,15693468,Simmons,488,Spain,0,39,9,140553.46,1,0,0,0
7679,7680,15790689,Hibbins,647,Spain,1,32,9,80958.36,1,1,1,0
4780,4781,15680046,Onochie,711,Spain,1,36,8,0.00,2,1,0,0


In [10]:
## Build the model
model=Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1) ## output layer for regression
])

##compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

##summary of the model
model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 3009 (11.75 KB)
Trainable params: 3009 (11.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [11]:
from datetime import datetime

In [12]:
### Set up the tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir="log/fit"+datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [13]:
### Set up Early Stopping
early_stoppping_callback=EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [14]:
print(x_train.dtypes)
print(y_train.dtypes)

RowNumber           int64
CustomerId          int64
Surname               str
CreditScore         int64
Geography             str
Gender              int32
Age                 int64
Tenure              int64
Balance           float64
NumOfProducts       int64
HasCrCard           int64
IsActiveMember      int64
Exited              int64
dtype: object
float64


In [15]:
print(x_train.select_dtypes(include='object').columns)

Index(['Surname', 'Geography'], dtype='str')


C:\Users\sbraj\AppData\Local\Temp\ipykernel_22412\3463681098.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(x_train.select_dtypes(include='object').columns)


In [16]:
x_train = x_train.drop(columns=['Surname'])
x_test = x_test.drop(columns=['Surname'])

In [17]:
x_train = pd.get_dummies(x_train, columns=['Geography'], drop_first=True)
x_test = pd.get_dummies(x_test, columns=['Geography'], drop_first=True)

In [18]:
x_train, x_test = x_train.align(x_test, join='left', axis=1, fill_value=0)

In [19]:
x_train = x_train.astype("float32")
x_test = x_test.astype("float32")
y_train = y_train.astype("float32")
y_test = y_test.astype("float32")

In [20]:
### Training the Model
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=100,
    callbacks=[tensorflow_callback, early_stoppping_callback]
)

Epoch 1/100


219/219 [==============================] - 2s 4ms/step - loss: 61657.4766 - mae: 61657.4766 - val_loss: 49767.6211 - val_mae: 49767.6211
Epoch 2/100
219/219 [==============================] - 1s 3ms/step - loss: 54174.3711 - mae: 54174.3711 - val_loss: 49940.8477 - val_mae: 49940.8477
Epoch 3/100
219/219 [==============================] - 1s 3ms/step - loss: 57404.8242 - mae: 57404.8242 - val_loss: 67094.6953 - val_mae: 67094.6953
Epoch 4/100
219/219 [==============================] - 1s 3ms/step - loss: 55110.2734 - mae: 55110.2734 - val_loss: 61960.8438 - val_mae: 61960.8438
Epoch 5/100
219/219 [==============================] - 1s 2ms/step - loss: 55149.5391 - mae: 55149.5391 - val_loss: 50142.3750 - val_mae: 50142.3750
Epoch 6/100
219/219 [==============================] - 1s 3ms/step - loss: 52781.8633 - mae: 52781.8633 - val_loss: 54226.2070 - val_mae: 54226.2070
Epoch 7/100
219/219 [==============================] - 1s 3ms/step - loss: 55614.6875 - mae: 55614.6875 

In [24]:
import sys
!{sys.executable} -m pip install --upgrade pip setuptools wheel

In [25]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [26]:
%tensorboard --logdir regressionlogs/fit

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "D:\DeepLearning\venv\Scripts\tensorboard.exe\__main__.py", line 2, in <module>
  File "D:\DeepLearning\venv\Lib\site-packages\tensorboard\main.py", line 27, in <module>
    from tensorboard import default
  File "D:\DeepLearning\venv\Lib\site-packages\tensorboard\default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'

In [27]:
##evaluate model
test_loss,test_mae=model.evaluate(x_test,y_test)


94/94 [==============================] - 0s 1ms/step - loss: 49767.6211 - mae: 49767.6211


In [29]:
model.save('regression_model.h5')

d:\DeepLearning\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
